### This was done in August, 2026 using PostgreSQL.

# Practical Exam: Hotel Operations

LuxurStay Hotels is a major, international chain of hotels. They offer hotels for both business and leisure travellers in major cities across the world. The chain prides themselves on the level of customer service that they offer. 

However, the management has been receiving complaints about slow room service in some hotel branches. As these complaints are impacting the customer satisfaction rates, it has become a serious issue. Recent data shows that customer satisfaction has dropped from the 4.5 rating that they expect. 

You are working with the Head of Operations to identify possible causes and hotel branches with the worst problems. 

## Data
The following schema diagram shows the tables available. You have only been provided with data where customers provided a feedback rating (reference the `hotel_operations.png` file).

![hotel_operations](hotel_operations.png)

## Exam Guidelines
- Use SQL to perform each of the tasks.
- Write your solutions in the code cell provided. Each solution must be in a **SINGLE** cell.
- The final output of your query will be graded, not the code.
- Ensure you match any column name requirements.
- You must be successful in all tasks to pass this exam.
- You will only have **TWO** opportunities to submit your exam for grading. 

## Summary of Analyses

LuxuryStars Hotels, in the interest of solving the growing decline in customer dissatisfaction due to slow room service, wants to launch an investigation to identify possible causes & hotel branches with the worst problems. This project explored data revolving around the hotel company's procedural & logistical information using three tables: **Services** indicating particular services the hotels perform; **Requests** indicating sent by customers including the corresponding service & branch, the time taken until a response, & the rating of the response; **Branches** indicating details of each branch like the location, total rooms, staff counts, opening date, & types of guests expected to use the hotel. There are four unique services, 100 unique branches, & over 17,500 unique customer requests.

- **Task 1** explored & cleaned the data in the `branch` table including problems like typos, inconsistent values, missing values, & incorrect data types.
- In **Task 2**, the average & maximum amounts of time taken for customer requests to get responded to were analyzed across the four services & 100 branches.
- In **Task 3**, data was aggregated & then condensed to look at particular services & branch locations specifically Meal & Laundry services in hotels in Europe & Latin America. There are 5,047 such instances.
- In **Task 4**, the lowest performing hotels were obtained based on an average rating of 4.5 (a target set by management) across the different services & branches. There were 215 service-branch pairs of hotels that had an average request rating less than 4.50.

> 

*** Unfortunately, this exam was submitted before the results could be properly dissected & so findings & recommendations are not available.

# Task 1

Before you can start any analysis, you need to confirm that the data is accurate and reflects what you expect to see. 

It is known that there are some issues with the `branch` table, and the data team have provided the following data description. 

Write a query to return data matching this description, including identifying and cleaning all invalid values. You must match all column names and description criteria. Your output should be a DataFrame named 'clean_branch_data'.

| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
|id | Nominal. The unique identifier of the hotel. </br>Missing values are not possible due to the database structure.|
| location | Nominal. The location of the particular hotel. One of four possible values, 'EMEA', 'NA', 'LATAM' and 'APAC'. </br>Missing values should be replaced with “Unknown”. |
| total_rooms | Discrete. The total number of rooms in the hotel. Must be a positive integer between 1 and 400. </br>Missing values should be replaced with the default number of rooms, 100. |
| staff_count | Discrete. The number of staff employeed in the hotel service department. </br>Missing values should be replaced with the total_rooms multiplied by 1.5. |
| opening_date | Discrete. The year in which the hotel opened. This can be any value between 2000 and 2023. </br>Missing values should be replaced with 2023. |
| target_guests | Nominal. The primary type of guest that is expected to use the hotel. Can be one of 'Leisure' or 'Business'. </br>Missing values should be replaced with 'Leisure'. |

#### Plan of Action:
- Evaluate the `branch` table to determine what needs preprocessing.
    - Table has 100 rows.
    - Each variable was checked to find inconsistencies & such.
        - 'id': INT. 100 unique values, no missing values. GOOD.
        - 'location': TEXT. Appropriate values, no extra whitespaces, no missing values. GOOD.
        - 'total_rooms': INT. Values are between 1-400. 10 rows with missing values; need to change to 100.
        - 'staff_count': INT. No missing values. GOOD.
        - 'opening_date': TEXT. Values are between 2000-2023. Are 4 rows with missing values ("-"); need to change to 2023. Should change type to INT.
        - 'target_guests': TEXT. Are multiple typos ("B.", "Busniess"); need to change to "Business".
- Clean data -- missing data, incorrect values, duplicates.
- Fix data types.
    - In PostgreSQL, use PG_TYPEOF(_variable_) to check data types.
- Recheck cleaned data.

In [ ]:
#Task 1 query

#Check data types of columns
SELECT *
FROM INFORMATION_SCHEMA.columns
WHERE table_name = 'branch';

#Investigate values of columns
SELECT id, COUNT(id)
FROM branch
GROUP BY id
ORDER BY id;

#Check for NULL values
SELECT *
FROM branch
WHERE total_rooms IS NULL;

#Adjust columns as necessary
    # 'total_rooms' -- Replace null values with 100
SELECT COALESCE(total_rooms, 100) AS total_rooms ...
    # 'opening_date' -- Replace missing values ("-") with 2023. Change data types from TEXT to INT.
SELECT REPLACE(opening_date, '-', '2023')::INT AS opening_date ...
    # 'target_guests' -- Correct inconsistencies ("B.", "Busniess" to "Business")
SELECT CASE WHEN target_guests = 'B.' THEN 'Business'
        WHEN target_guests = 'Busniess' THEN 'Business'
        WHEN target_guests = 'Business' THEN 'Business'
        ELSE 'Leisure' END AS target_guests


#Execute adjustments, check results
WITH task1_clean AS (
    SELECT id, location, 
    COALESCE(total_rooms, 100) AS total_rooms,
    staff_count,
    REPLACE(opening_date, '-', '2023')::INT AS opening_date,
    CASE WHEN target_guests = 'B.' THEN 'Business'
        WHEN target_guests = 'Busniess' THEN 'Business'
        WHEN target_guests = 'Business' THEN 'Business'
        ELSE 'Leisure' END AS target_guests
    FROM branch )

SELECT DISTINCT PG_TYPEOF(opening_date), *
FROM task1_clean
WHERE total_rooms IS NULL;

#Output final query (returns 100 rows)
SELECT id, lcoation,
    COALESCE(total_rooms, 100) AS total_rooms,
    staff_count,
    REPLACE(opening_date, '-', '2023')::INT AS opening_date,
    CASE WHEN target_guests = 'B.' THEN 'Business'
        WHEN target_guests = 'Busniess' THEN 'Business'
        WHEN target_guests = 'Business' THEN 'Business'
        ELSE 'Leisure' END AS target_guests 
FROM branch;

# Task 2
The Head of Operations wants to know whether there is a difference in time taken to respond to a customer request in each hotel. They already know that different services take different lengths of time.  
Calculate the average and maximum duration for each branch and service. Your output should include the columns `service_id`, `branch_id`, `avg_time_taken` and `max_time_taken`. Values should be rounded to two decimal places where appropriate.


#### Plan of Action:
- There are 17,682 rows in the `request` table.
- Need to group by 'service_id' & 'branch_id' then calculate the average & max 'time_taken' to respond to customer requests.
- Need to round numeric values to 2 decimals.

In [ ]:
#Task 2 query

SELECT service_id, branch_id, ROUND(AVG(time_taken), 2) AS avg_time_taken,
    ROUND(MAX(time_taken), 2) AS max_time_taken
FROM request
GROUP BY service_id, branch_id;
    #returns 385 rows

# Task 3
The management team want to target improvements in `Meal` and `Laundry` service in Europe (`EMEA`) and Latin America (`LATAM`).

Write a query to return the `description` of the service, the `id` and `location` of the branch, the id of the request as `request_id` and the `rating` for the services and locations of interest to the management team.  
Use the original `branch` table, not the output of task 1.


#### Plan of Action:
- There are 100 & 4 rows in the `branch` & `service` tables respectively.
- Need to join the 3 tables together using inner joins.
- Need to apply filters. First, filter to "Meal" & "Laundry" services using the 'description' field in the `service` table. Then, filter to 'EMEA' & 'LATAM' locations using the 'location' field in the `branch` table.
- For variables with overlapping names in different tables, need to specify the corresponding table.

In [ ]:
#Task 3 query

SELECT description, branch.id, location, request.id AS request_id, rating
FROM service
INNER JOIN request
ON service.id = request.service_id
INNER JOIN branch
ON request.branch_id = branch.id
WHERE description IN ('Meal','Laundry') AND location IN ('EMEA','LATAM');
    #returns 5,047 rows

# Task 4
So that you can take a more detailed look at the lowest performing hotels, you want to get service and branch information where the average rating for the branch and service combination is lower than 4.5 - the target set by management.  
Your query should return the `service_id` and `branch_id`, and the average rating (`avg_rating`), rounded to 2 decimal places.



#### Plan of Action:
- Recall that there are 17,682 rows.
- Need to calculate the average 'rating' for each 'service_id'-'branch_id' pair then filter for those with an average rating below 4.50.
- Need to round the averages to 2 decimals.

In [ ]:
#Task 4 query

SELECT service_id, branch_id, ROUND(AVG(rating), 2) AS avg_rating
FROM request
GROUP BY service_id, branch_id
HAVING AVG(rating) < 4.50;
    #returns 215 rows